In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.neighbors import BallTree
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ── Geohash decoder ────────────────────────────────────────────────────────────
def geohash_decode(geohash):
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_range = (-90.0, 90.0); lon_range = (-180.0, 180.0); is_lon = True
    for char in geohash:
        bits = base32.index(char)
        for i in range(4, -1, -1):
            bit = (bits >> i) & 1
            if is_lon:
                mid = (lon_range[0]+lon_range[1])/2
                lon_range = (mid,lon_range[1]) if bit else (lon_range[0],mid)
            else:
                mid = (lat_range[0]+lat_range[1])/2
                lat_range = (mid,lat_range[1]) if bit else (lat_range[0],mid)
            is_lon = not is_lon
    return (lat_range[0]+lat_range[1])/2, (lon_range[0]+lon_range[1])/2

print("Loading data...")
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

all_geo = pd.Series(pd.concat([train['geohash'], test['geohash']]).unique())
geo_coords = pd.DataFrame({'geohash': all_geo})
geo_coords[['lat','lon']] = geo_coords['geohash'].apply(
    lambda g: pd.Series(geohash_decode(g)))

ref_median  = train['Temperature'].median()
global_mean = train['demand'].mean()

# ── Spatial neighbor features ──────────────────────────────────────────────────
geo_demand = train.groupby('geohash')['demand'].mean().reset_index()
geo_demand.columns = ['geohash', 'geo_mean_all']
gc2 = geo_coords.merge(geo_demand, on='geohash', how='left')
gc2['geo_mean_all'] = gc2['geo_mean_all'].fillna(global_mean)

bt = BallTree(np.radians(gc2[['lat','lon']].values), metric='haversine')
# k=11 for distance-weighted neighbors
dists, idxs = bt.query(np.radians(gc2[['lat','lon']].values), k=11)
neighbor_vals = gc2['geo_mean_all'].values[idxs[:,1:]]
weights = 1.0 / (dists[:,1:] + 1e-6)
weights /= weights.sum(axis=1, keepdims=True)
gc2['neighbor_geo_mean_wt'] = (neighbor_vals * weights).sum(axis=1)
gc2['neighbor_geo_mean']    = neighbor_vals.mean(axis=1)
gc2['neighbor_geo_mean_k1'] = gc2['geo_mean_all'].values[idxs[:,1]]
gc2['neighbor_geo_std']     = neighbor_vals.std(axis=1)
geo_coords = geo_coords.merge(
    gc2[['geohash','neighbor_geo_mean','neighbor_geo_mean_wt',
          'neighbor_geo_mean_k1','neighbor_geo_std']], on='geohash', how='left')

# ── Day-49 early-hour features (hours 0, 1, 2) ───────────────────────────────
day49_train = train[train['day']==49].copy()
day49_train['hour_int'] = day49_train['timestamp'].apply(lambda x: int(x.split(':')[0]))

day49_h0 = (day49_train[day49_train['hour_int']==0]
            .groupby('geohash')['demand'].mean().reset_index()
            .rename(columns={'demand':'day49_h0_mean'}))
day49_h1 = (day49_train[day49_train['hour_int']==1]
            .groupby('geohash')['demand'].mean().reset_index()
            .rename(columns={'demand':'day49_h1_mean'}))
day49_h2 = (day49_train[day49_train['hour_int']==2]
            .groupby('geohash')['demand'].mean().reset_index()
            .rename(columns={'demand':'day49_h2_mean'}))
day49_early = (day49_train.groupby('geohash')['demand'].mean().reset_index()
               .rename(columns={'demand':'day49_early_mean'}))
# Day49 early variance per geohash
day49_early_std = (day49_train.groupby('geohash')['demand'].std().reset_index()
                   .rename(columns={'demand':'day49_early_std'}))
day49_early_max = (day49_train.groupby('geohash')['demand'].max().reset_index()
                   .rename(columns={'demand':'day49_early_max'}))

# ── Day-48 per-geohash early hour stats (for ratio feature) ───────────────────
day48_train = train[train['day']==48].copy()
day48_train['hour_int'] = day48_train['timestamp'].apply(lambda x: int(x.split(':')[0]))
day48_early_geo = (day48_train[day48_train['hour_int']<=2]
                   .groupby('geohash')['demand'].mean().reset_index()
                   .rename(columns={'demand':'d48_early_geo'}))
# Day48 hours 2-13 mean per geohash (matches test period)
day48_test_hours_geo = (day48_train[(day48_train['hour_int']>=2) & (day48_train['hour_int']<=13)]
                        .groupby('geohash')['demand'].mean().reset_index()
                        .rename(columns={'demand':'d48_test_period_geo_mean'}))

# ── Static features per geohash (modal value) ─────────────────────────────────
# Some geohashes have varying RoadType etc. Use modal value as a stable feature
for col in ['RoadType', 'LargeVehicles', 'Landmarks']:
    modal = (train.groupby('geohash')[col]
             .agg(lambda x: x.mode()[0] if len(x.dropna()) > 0 else np.nan)
             .reset_index().rename(columns={col: f'{col}_modal'}))
    geo_coords = geo_coords.merge(modal, on='geohash', how='left')

modal_lanes = (train.groupby('geohash')['NumberofLanes']
               .agg(lambda x: x.mode()[0] if len(x) > 0 else np.nan)
               .reset_index().rename(columns={'NumberofLanes': 'NumberofLanes_modal'}))
geo_coords = geo_coords.merge(modal_lanes, on='geohash', how='left')

# ── Temperature per (geohash, hour) from day48 ───────────────────────────────
# Used to impute missing temp in test
day48_temp = day48_train[['geohash','timestamp','Temperature']].rename(
    columns={'Temperature': 'd48_temp'})
# Also geohash-hour mean temperature from day48
day48_train['hour_int2'] = day48_train['timestamp'].apply(lambda x: int(x.split(':')[0]))
d48_geo_hourtemp = (day48_train.groupby(['geohash','hour_int2'])['Temperature']
                    .mean().reset_index()
                    .rename(columns={'Temperature': 'd48_geo_hourtemp', 'hour_int2': 'hour_for_temp'}))

# ── Base feature engineering ───────────────────────────────────────────────────
def base_features(df, ref_temp_median):
    df = df.copy()
    df['hour']        = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute']      = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_of_day'] = df['hour']*60 + df['minute']

    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['time_sin'] = np.sin(2*np.pi*df['time_of_day']/(24*60))
    df['time_cos'] = np.cos(2*np.pi*df['time_of_day']/(24*60))
    df['min_sin']  = np.sin(2*np.pi*df['minute']/60)
    df['min_cos']  = np.cos(2*np.pi*df['minute']/60)

    df['is_morning_rush'] = ((df['hour']>=7) &(df['hour']<=9) ).astype(int)
    df['is_evening_rush'] = ((df['hour']>=17)&(df['hour']<=19)).astype(int)
    df['is_night']        = ((df['hour']>=23)|(df['hour']<=5) ).astype(int)
    df['is_midday']       = ((df['hour']>=11)&(df['hour']<=13)).astype(int)
    df['is_early_morn']   = ((df['hour']>=2) &(df['hour']<=4) ).astype(int)

    df['RoadType_enc']      = df['RoadType'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1)
    df['LargeVehicles_enc'] = (df['LargeVehicles']=='Allowed').astype(int)
    df['Landmarks_enc']     = (df['Landmarks']=='Yes').astype(int)
    df['Weather_enc']       = df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}).fillna(-1)

    # Better temperature handling: impute from day48 same (geohash, timestamp) then hour-level
    df['hour_for_temp'] = df['hour']
    df = df.merge(day48_temp, on=['geohash','timestamp'], how='left')
    df = df.merge(d48_geo_hourtemp, on=['geohash','hour_for_temp'], how='left')
    df['Temperature_filled'] = df['Temperature'].copy()
    # Fill test missing temp: first from d48 same slot, then d48 geo+hour mean, then global median
    df['Temperature_filled'] = df['Temperature_filled'].fillna(df['d48_temp'])
    df['Temperature_filled'] = df['Temperature_filled'].fillna(df['d48_geo_hourtemp'])
    df['Temperature_filled'] = df['Temperature_filled'].fillna(ref_temp_median)
    df['temp_missing']        = df['Temperature'].isna().astype(int)
    df['temp_x_weather']      = df['Temperature_filled'] * (df['Weather_enc']+1)
    df['temp_sq']             = df['Temperature_filled'] ** 2
    # Temperature difference from global median
    df['temp_anomaly']        = df['Temperature_filled'] - ref_temp_median
    df.drop(columns=['d48_temp', 'd48_geo_hourtemp', 'hour_for_temp'], inplace=True)

    df['geo_prefix3'] = df['geohash'].str[:3]
    df['geo_prefix4'] = df['geohash'].str[:4]
    df['geo_prefix5'] = df['geohash'].str[:5]

    df['lanes_road']     = df['NumberofLanes'] * (df['RoadType_enc']+2)
    df['lanes_landmark'] = df['NumberofLanes'] * df['Landmarks_enc']
    df['highway_rush']   = (df['RoadType_enc']==2).astype(int) * (
                            df['is_morning_rush'] + df['is_evening_rush'])
    df['is_highway']     = (df['RoadType_enc']==2).astype(int)
    df['weather_x_rush'] = df['Weather_enc'] * (df['is_morning_rush'] + df['is_evening_rush'])
    df['lanes_x_rush']   = df['NumberofLanes'] * (df['is_morning_rush'] + df['is_evening_rush'])

    df = df.merge(geo_coords, on='geohash', how='left')
    df = df.merge(day49_h0,        on='geohash', how='left')
    df = df.merge(day49_h1,        on='geohash', how='left')
    df = df.merge(day49_h2,        on='geohash', how='left')
    df = df.merge(day49_early,     on='geohash', how='left')
    df = df.merge(day49_early_std, on='geohash', how='left')
    df = df.merge(day49_early_max, on='geohash', how='left')
    df = df.merge(day48_early_geo, on='geohash', how='left')
    df = df.merge(day48_test_hours_geo, on='geohash', how='left')

    # Encode modal static features
    df['RoadType_modal_enc']     = df['RoadType_modal'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1)
    df['LargeVehicles_modal_enc']= (df['LargeVehicles_modal']=='Allowed').astype(int)
    df['Landmarks_modal_enc']    = (df['Landmarks_modal']=='Yes').astype(int)

    return df

print("Engineering features...")
train_fe = base_features(train, ref_median)
test_fe  = base_features(test,  ref_median)

# Zero out day49 early features for day49 train rows (prevent leakage for hours 0-2)
for col in ['day49_h0_mean','day49_h1_mean','day49_h2_mean','day49_early_mean',
            'day49_early_std','day49_early_max']:
    train_fe.loc[train_fe['day']==49, col] = np.nan

# ── OOF target encodings (no leakage) ─────────────────────────────────────────
print("Computing OOF target encodings...")
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

agg_keys_list = [
    (['geohash', 'time_of_day'],          'geo_time_mean'),
    (['geohash', 'hour'],                 'geo_hour_mean'),
    (['geohash'],                         'geo_mean'),
    (['geo_prefix5'],                     'prefix5_mean'),
    (['geo_prefix4'],                     'prefix4_mean'),
    (['geo_prefix3'],                     'prefix3_mean'),
    (['time_of_day'],                     'time_global_mean'),
    (['NumberofLanes','time_of_day'],     'lanes_time_mean'),
    (['geohash','Weather_enc'],           'geo_weather_mean'),
    (['geohash','RoadType_enc'],          'geo_road_mean'),
    (['geo_prefix4','hour'],              'prefix4_hour_mean'),
    (['geo_prefix4','time_of_day'],       'prefix4_time_mean'),
    (['RoadType_enc','hour'],             'road_hour_mean'),
    (['LargeVehicles_enc','hour'],        'largeveh_hour_mean'),
    (['Landmarks_enc','time_of_day'],     'landmark_time_mean'),
    (['NumberofLanes','hour'],            'lanes_hour_mean'),
    (['RoadType_enc','time_of_day'],      'road_time_mean'),
    # NEW encodings
    (['geohash','day'],                   'geo_day_mean'),       # day-level geo trend
    (['geo_prefix4','Weather_enc'],       'prefix4_weather_mean'),
    (['geohash','is_morning_rush'],       'geo_rush_mean'),
    (['geohash','is_night'],              'geo_night_mean'),
    (['NumberofLanes','RoadType_enc'],    'lanes_roadtype_mean'),
    (['geo_prefix5','hour'],              'prefix5_hour_mean'),
    (['geo_prefix3','time_of_day'],       'prefix3_time_mean'),
    (['RoadType_enc','Weather_enc'],      'road_weather_mean'),
    (['geohash','is_midday'],             'geo_midday_mean'),
    (['Landmarks_enc','hour'],            'landmark_hour_mean'),
    (['geo_prefix4','RoadType_enc'],      'prefix4_road_mean'),
]

for _, col in agg_keys_list:
    train_fe[col] = np.nan

for fold, (tr_idx, val_idx) in enumerate(kf5.split(train_fe)):
    tr = train_fe.iloc[tr_idx]
    for keys, col in agg_keys_list:
        agg = tr.groupby(keys)['demand'].mean().reset_index().rename(columns={'demand': col})
        merged = train_fe.iloc[val_idx][keys].merge(agg, on=keys, how='left')
        train_fe.loc[val_idx, col] = merged[col].values
    print(f"  fold {fold+1}/5 done")

for _, col in agg_keys_list:
    train_fe[col] = train_fe[col].fillna(global_mean)

for keys, col in agg_keys_list:
    agg = train_fe.groupby(keys)['demand'].mean().reset_index().rename(
        columns={'demand': col+'_t'})
    test_fe = test_fe.merge(agg, on=keys, how='left')
    test_fe[col] = test_fe[col+'_t'].fillna(global_mean)
    test_fe.drop(columns=[col+'_t'], inplace=True)

# ── OOF geo stats ─────────────────────────────────────────────────────────────
for col_nm in ['geo_std','geo_median','geo_p25','geo_p75']:
    train_fe[col_nm] = np.nan

for fold, (tr_idx, val_idx) in enumerate(kf5.split(train_fe)):
    tr = train_fe.iloc[tr_idx]
    gs = tr.groupby('geohash')['demand'].agg(
        ['std','median',
         lambda x: x.quantile(0.25),
         lambda x: x.quantile(0.75)]
    ).reset_index()
    gs.columns = ['geohash','geo_std','geo_median','geo_p25','geo_p75']
    merged = train_fe.iloc[val_idx][['geohash']].merge(gs, on='geohash', how='left')
    for col_nm in ['geo_std','geo_median','geo_p25','geo_p75']:
        train_fe.loc[val_idx, col_nm] = merged[col_nm].values

train_fe['geo_std']    = train_fe['geo_std'].fillna(0)
train_fe['geo_median'] = train_fe['geo_median'].fillna(global_mean)
train_fe['geo_p25']    = train_fe['geo_p25'].fillna(global_mean)
train_fe['geo_p75']    = train_fe['geo_p75'].fillna(global_mean)

gs_full = train_fe.groupby('geohash')['demand'].agg(
    ['std','median',
     lambda x: x.quantile(0.25),
     lambda x: x.quantile(0.75)]
).reset_index()
gs_full.columns = ['geohash','geo_std','geo_median','geo_p25','geo_p75']
test_fe = test_fe.merge(gs_full, on='geohash', how='left')
for col_nm in ['geo_std','geo_median','geo_p25','geo_p75']:
    test_fe[col_nm] = test_fe[col_nm].fillna(global_mean if col_nm != 'geo_std' else 0)

train_fe['geo_cv']  = train_fe['geo_std'] / (train_fe['geo_mean'] + 1e-8)
test_fe['geo_cv']   = test_fe['geo_std']  / (test_fe['geo_mean']  + 1e-8)
train_fe['geo_iqr'] = train_fe['geo_p75'] - train_fe['geo_p25']
test_fe['geo_iqr']  = test_fe['geo_p75']  - test_fe['geo_p25']

# ── Lag: day 48 exact lookup ───────────────────────────────────────────────────
day48 = (train[train['day']==48][['geohash','timestamp','demand']]
         .rename(columns={'demand':'lag_day48'}))
train_fe = train_fe.merge(day48, on=['geohash','timestamp'], how='left')
test_fe  = test_fe.merge(day48, on=['geohash','timestamp'], how='left')
train_fe['lag_day48'] = train_fe['lag_day48'].fillna(train_fe['geo_time_mean'])
test_fe['lag_day48']  = test_fe['lag_day48'].fillna(test_fe['geo_time_mean'])

# ── Rolling / shifted lag from day 48 ────────────────────────────────────────
day48s = day48.copy()
day48s['time_of_day'] = day48s['timestamp'].apply(
    lambda x: int(x.split(':')[0])*60 + int(x.split(':')[1]))
day48s = day48s.sort_values(['geohash','time_of_day'])
day48s['lag48_roll4']  = day48s.groupby('geohash')['lag_day48'].transform(
    lambda x: x.rolling(4,  min_periods=1).mean())
day48s['lag48_roll8']  = day48s.groupby('geohash')['lag_day48'].transform(
    lambda x: x.rolling(8,  min_periods=1).mean())
day48s['lag48_roll12'] = day48s.groupby('geohash')['lag_day48'].transform(
    lambda x: x.rolling(12, min_periods=1).mean())
day48s['lag48_shift4'] = (day48s.groupby('geohash')['lag_day48']
                          .shift(4).fillna(day48s['lag_day48']))
# NEW: more rolling windows
day48s['lag48_roll2']  = day48s.groupby('geohash')['lag_day48'].transform(
    lambda x: x.rolling(2,  min_periods=1).mean())
day48s['lag48_roll16'] = day48s.groupby('geohash')['lag_day48'].transform(
    lambda x: x.rolling(16, min_periods=1).mean())
# Forward shift (next time step from day48 as feature)
day48s['lag48_shift_n1'] = (day48s.groupby('geohash')['lag_day48']
                             .shift(-1).fillna(day48s['lag_day48']))
# Day48 previous slot - shift +1 (exact 15min before)
day48s['lag48_shift1'] = (day48s.groupby('geohash')['lag_day48']
                          .shift(1).fillna(day48s['lag_day48']))

roll_feats = day48s[['geohash','timestamp',
                      'lag48_roll2','lag48_roll4','lag48_roll8',
                      'lag48_roll12','lag48_roll16',
                      'lag48_shift1','lag48_shift4','lag48_shift_n1']]

train_fe = train_fe.merge(roll_feats, on=['geohash','timestamp'], how='left')
test_fe  = test_fe.merge(roll_feats,  on=['geohash','timestamp'], how='left')
for col in ['lag48_roll2','lag48_roll4','lag48_roll8','lag48_roll12','lag48_roll16',
            'lag48_shift1','lag48_shift4','lag48_shift_n1']:
    train_fe[col] = train_fe[col].fillna(train_fe['geo_time_mean'])
    test_fe[col]  = test_fe[col].fillna(test_fe['geo_time_mean'])

# ── Day49 ratio feature: d49_early / d48_early per geohash ───────────────────
# Captures day-level scaling factor (how is today different from yesterday)
d49_early_ratio_df = day49_early.merge(day48_early_geo, on='geohash', how='left')
d49_early_ratio_df['d49_d48_ratio'] = np.clip(
    d49_early_ratio_df['day49_early_mean'] / (d49_early_ratio_df['d48_early_geo'] + 1e-6),
    0.1, 10.0)
d49_early_ratio_df = d49_early_ratio_df[['geohash','d49_d48_ratio']]

train_fe = train_fe.merge(d49_early_ratio_df, on='geohash', how='left')
test_fe  = test_fe.merge(d49_early_ratio_df,  on='geohash', how='left')
train_fe['d49_d48_ratio'] = train_fe['d49_d48_ratio'].fillna(1.0)
test_fe['d49_d48_ratio']  = test_fe['d49_d48_ratio'].fillna(1.0)

# Zero out ratio for day49 train rows (would leak future)
train_fe.loc[train_fe['day']==49, 'd49_d48_ratio'] = 1.0

# Ratio-adjusted lag: what day48 suggests scaled by today's early behavior
train_fe['lag48_ratio_adj']  = train_fe['lag_day48'] * train_fe['d49_d48_ratio']
test_fe['lag48_ratio_adj']   = test_fe['lag_day48']  * test_fe['d49_d48_ratio']
# Similarly adjust rolling features
train_fe['roll4_ratio_adj']  = train_fe['lag48_roll4'] * train_fe['d49_d48_ratio']
test_fe['roll4_ratio_adj']   = test_fe['lag48_roll4']  * test_fe['d49_d48_ratio']
train_fe['roll8_ratio_adj']  = train_fe['lag48_roll8'] * train_fe['d49_d48_ratio']
test_fe['roll8_ratio_adj']   = test_fe['lag48_roll8']  * test_fe['d49_d48_ratio']

# ── Fix day49 early features (now geo_time_mean is available) ─────────────────
for col in ['day49_h0_mean','day49_h1_mean','day49_h2_mean',
            'day49_early_mean','day49_early_std','day49_early_max']:
    train_fe.loc[train_fe['day']==49, col] = (
        train_fe.loc[train_fe['day']==49, 'geo_time_mean'])
    train_fe[col] = train_fe[col].fillna(train_fe['geo_mean'])
    test_fe[col]  = test_fe[col].fillna(test_fe['geo_mean'])

# ── Ratio features ─────────────────────────────────────────────────────────────
train_fe['lag48_to_geo_mean']     = train_fe['lag_day48']      / (train_fe['geo_mean'] + 1e-8)
test_fe['lag48_to_geo_mean']      = test_fe['lag_day48']       / (test_fe['geo_mean']  + 1e-8)
train_fe['geo_time_to_geo']       = train_fe['geo_time_mean']  / (train_fe['geo_mean'] + 1e-8)
test_fe['geo_time_to_geo']        = test_fe['geo_time_mean']   / (test_fe['geo_mean']  + 1e-8)
# NEW ratio features
train_fe['lag48_to_geo_hour']     = train_fe['lag_day48']      / (train_fe['geo_hour_mean'] + 1e-8)
test_fe['lag48_to_geo_hour']      = test_fe['lag_day48']       / (test_fe['geo_hour_mean']  + 1e-8)
train_fe['geo_hour_to_geo']       = train_fe['geo_hour_mean']  / (train_fe['geo_mean'] + 1e-8)
test_fe['geo_hour_to_geo']        = test_fe['geo_hour_mean']   / (test_fe['geo_mean']  + 1e-8)
train_fe['d49_early_to_d48_early']= train_fe['day49_early_mean'] / (train_fe['d48_early_geo'] + 1e-8)
test_fe['d49_early_to_d48_early'] = test_fe['day49_early_mean']  / (test_fe['d48_early_geo']  + 1e-8)
train_fe['lag48_to_time_global']  = train_fe['lag_day48']      / (train_fe['time_global_mean'] + 1e-8)
test_fe['lag48_to_time_global']   = test_fe['lag_day48']       / (test_fe['time_global_mean']  + 1e-8)

# ── Per-hour geohash stats from day48 ─────────────────────────────────────────
# Day48 per (geohash, hour): mean, std, max
for stat_name, agg_fn in [('mean','mean'), ('std','std'), ('max','max')]:
    d48_gh_stat = (day48_train.groupby(['geohash','hour_int'])['demand']
                   .agg(agg_fn).reset_index()
                   .rename(columns={'demand': f'd48_geo_hour_{stat_name}', 'hour_int': 'hour'}))
    train_fe = train_fe.merge(d48_gh_stat, on=['geohash','hour'], how='left')
    test_fe  = test_fe.merge(d48_gh_stat,  on=['geohash','hour'], how='left')
    fill_val = global_mean if stat_name == 'mean' else 0
    train_fe[f'd48_geo_hour_{stat_name}'] = train_fe[f'd48_geo_hour_{stat_name}'].fillna(fill_val)
    test_fe[f'd48_geo_hour_{stat_name}']  = test_fe[f'd48_geo_hour_{stat_name}'].fillna(fill_val)

# Ratio-adjusted hour mean
train_fe['d48_geo_hour_mean_radj'] = train_fe['d48_geo_hour_mean'] * train_fe['d49_d48_ratio']
test_fe['d48_geo_hour_mean_radj']  = test_fe['d48_geo_hour_mean']  * test_fe['d49_d48_ratio']

# ── Prefix-level time-of-day std / max from day48 ────────────────────────────
for prefix_col in ['geo_prefix5', 'geo_prefix4']:
    d48_train_p = day48_train.copy()
    d48_train_p['time_of_day'] = d48_train_p['hour_int']*60 + d48_train_p['timestamp'].apply(lambda x: int(x.split(':')[1]))
    d48_train_p[prefix_col] = d48_train_p['geohash'].str[:int(prefix_col[-1])]
    p_std = (d48_train_p.groupby([prefix_col,'time_of_day'])['demand']
             .std().reset_index().rename(columns={'demand': f'{prefix_col}_tod_std'}))
    train_fe = train_fe.merge(p_std, on=[prefix_col,'time_of_day'], how='left')
    test_fe  = test_fe.merge(p_std, on=[prefix_col,'time_of_day'], how='left')
    train_fe[f'{prefix_col}_tod_std'] = train_fe[f'{prefix_col}_tod_std'].fillna(0)
    test_fe[f'{prefix_col}_tod_std']  = test_fe[f'{prefix_col}_tod_std'].fillna(0)

# ── Final feature list ─────────────────────────────────────────────────────────
feature_cols = [
    # Time
    'hour','minute','time_of_day',
    'hour_sin','hour_cos','time_sin','time_cos','min_sin','min_cos',
    'is_morning_rush','is_evening_rush','is_night','is_midday','is_highway','is_early_morn',
    # Road / place
    'RoadType_enc','LargeVehicles_enc','Landmarks_enc','Weather_enc',
    'RoadType_modal_enc','LargeVehicles_modal_enc','Landmarks_modal_enc',
    'Temperature_filled','temp_missing','temp_x_weather','temp_sq','temp_anomaly',
    'NumberofLanes','NumberofLanes_modal','day',
    'lanes_road','lanes_landmark','highway_rush','weather_x_rush','lanes_x_rush',
    # Geo stats
    'geo_mean','geo_std','geo_median','geo_cv','geo_p25','geo_p75','geo_iqr',
    # Target encodings - original
    'geo_time_mean','geo_hour_mean',
    'prefix5_mean','prefix4_mean','prefix3_mean',
    'time_global_mean','lanes_time_mean',
    'geo_weather_mean','geo_road_mean',
    'prefix4_hour_mean','prefix4_time_mean',
    'road_hour_mean','largeveh_hour_mean','landmark_time_mean',
    'lanes_hour_mean','road_time_mean',
    # Target encodings - new
    'geo_day_mean','prefix4_weather_mean','geo_rush_mean','geo_night_mean',
    'lanes_roadtype_mean','prefix5_hour_mean','prefix3_time_mean',
    'road_weather_mean','geo_midday_mean','landmark_hour_mean','prefix4_road_mean',
    # Spatial
    'lat','lon','neighbor_geo_mean','neighbor_geo_mean_wt',
    'neighbor_geo_mean_k1','neighbor_geo_std',
    # Lag / rolling
    'lag_day48',
    'lag48_roll2','lag48_roll4','lag48_roll8','lag48_roll12','lag48_roll16',
    'lag48_shift1','lag48_shift4','lag48_shift_n1',
    # Day48 per-(geohash, hour) stats
    'd48_geo_hour_mean','d48_geo_hour_std','d48_geo_hour_max',
    'd48_geo_hour_mean_radj',
    # Prefix tod std
    'geo_prefix5_tod_std','geo_prefix4_tod_std',
    # Day48 period stats
    'd48_early_geo','d48_test_period_geo_mean',
    # Ratio features
    'lag48_to_geo_mean','geo_time_to_geo','lag48_to_geo_hour','geo_hour_to_geo',
    'd49_early_to_d48_early','lag48_to_time_global',
    # Day49 ratio and ratio-adjusted
    'd49_d48_ratio','lag48_ratio_adj','roll4_ratio_adj','roll8_ratio_adj',
    # Same-day early signal
    'day49_h0_mean','day49_h1_mean','day49_h2_mean',
    'day49_early_mean','day49_early_std','day49_early_max',
]

X_train = train_fe[feature_cols].values
y_train = train_fe['demand'].values
X_test  = test_fe[feature_cols].values

print(f"Feature count: {len(feature_cols)}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# ── Model configs ──────────────────────────────────────────────────────────────
hgb_configs = [
    dict(max_iter=800,  learning_rate=0.03, max_depth=10, min_samples_leaf=10,
         l2_regularization=0.5,  random_state=42),
    dict(max_iter=600,  learning_rate=0.05, max_depth=8,  min_samples_leaf=15,
         l2_regularization=1.0,  random_state=0),
    dict(max_iter=1000, learning_rate=0.02, max_depth=12, min_samples_leaf=5,
         l2_regularization=0.1,  random_state=7),
    dict(max_iter=1200, learning_rate=0.015,max_depth=11, min_samples_leaf=8,
         l2_regularization=0.3,  random_state=13),
    dict(max_iter=700,  learning_rate=0.04, max_depth=9,  min_samples_leaf=12,
         l2_regularization=0.7,  random_state=99),
    dict(max_iter=1500, learning_rate=0.02, max_depth=13, min_samples_leaf=3,
         l2_regularization=0.05, random_state=202),
    dict(max_iter=2000, learning_rate=0.01, max_depth=6,  min_samples_leaf=20,
         l2_regularization=1.5,  random_state=101),
    dict(max_iter=600,  learning_rate=0.05, max_depth=15, min_samples_leaf=5,
         l2_regularization=0.2,  random_state=404),
    # NEW HGB: more depth with moderate regularization
    dict(max_iter=1000, learning_rate=0.025, max_depth=14, min_samples_leaf=4,
         l2_regularization=0.15, random_state=55),
    dict(max_iter=1800, learning_rate=0.012, max_depth=10, min_samples_leaf=10,
         l2_regularization=0.8,  random_state=77),
    dict(max_iter=900,  learning_rate=0.035, max_depth=11, min_samples_leaf=7,
         l2_regularization=0.4,  random_state=123),
]

et_configs = [
    dict(n_estimators=200, max_depth=20,   min_samples_leaf=10, max_features=0.7,
         random_state=3,  n_jobs=-1),
    dict(n_estimators=200, max_depth=30,   min_samples_leaf=4,  max_features=0.75,
         random_state=17, n_jobs=-1),
    dict(n_estimators=200, max_depth=None, min_samples_leaf=6,  max_features=0.65,
         random_state=23, n_jobs=-1),
    dict(n_estimators=200, max_depth=35,   min_samples_leaf=3,  max_features=0.75,
         random_state=31, n_jobs=-1),
    # NEW ET configs
    dict(n_estimators=200, max_depth=25,   min_samples_leaf=5,  max_features=0.8,
         random_state=91, n_jobs=-1),
    dict(n_estimators=150, max_depth=40,   min_samples_leaf=2,  max_features=0.7,
         random_state=61, n_jobs=-1),
]

# Random Forest configs (different split criterion from ET)
rf_configs = [
    dict(n_estimators=200, max_depth=25, min_samples_leaf=5, max_features=0.7,
         random_state=42, n_jobs=-1),
    dict(n_estimators=200, max_depth=None, min_samples_leaf=3, max_features=0.6,
         random_state=7, n_jobs=-1),
]

all_models  = ([HistGradientBoostingRegressor(**c) for c in hgb_configs] +
               [ExtraTreesRegressor(**c)            for c in et_configs] +
               [RandomForestRegressor(**c)           for c in rf_configs])
model_names = ([f'HGB_{i+1}'  for i in range(len(hgb_configs))] +
               [f'ET_{i+1}'   for i in range(len(et_configs))] +
               [f'RF_{i+1}'   for i in range(len(rf_configs))])

# ── Train + OOF predict ────────────────────────────────────────────────────────
cv = KFold(n_splits=5, shuffle=True, random_state=42)
n_models   = len(all_models)
oof_preds  = np.zeros((len(y_train), n_models))
test_preds = np.zeros((len(X_test),  n_models))

for i, (m, name) in enumerate(zip(all_models, model_names)):
    print(f"[{i+1}/{n_models}] Training {name} ...", flush=True)
    oof_preds[:,i] = cross_val_predict(m, X_train, y_train, cv=cv, n_jobs=-1)
    m.fit(X_train, y_train)
    test_preds[:,i] = m.predict(X_test)
    r2 = r2_score(y_train, oof_preds[:,i])
    print(f"  OOF R²: {r2:.4f}")

# ── Optimise ensemble weights (15 random restarts) ────────────────────────────
def neg_r2(w):
    w = np.abs(w)/np.abs(w).sum()
    return -r2_score(y_train, oof_preds @ w)

best_r2 = -1
best_w  = None
for seed in range(15):
    rng = np.random.default_rng(seed)
    w0  = rng.dirichlet(np.ones(n_models))
    res = minimize(neg_r2, w0, method='Nelder-Mead',
                   options={'maxiter':100000,'xatol':1e-10,'fatol':1e-10})
    w   = np.abs(res.x)/np.abs(res.x).sum()
    r2  = r2_score(y_train, oof_preds @ w)
    if r2 > best_r2:
        best_r2 = r2
        best_w  = w

print(f"\n{'─'*60}")
for name, w in zip(model_names, best_w):
    print(f"  {name:<10}  weight = {w:.4f}")
print(f"{'─'*60}")
print(f"Optimised OOF R²  : {best_r2:.6f}")
print(f"Estimated score   : {100*best_r2:.4f} / 100")

# ── Stage 2: Meta-learner on OOF predictions ──────────────────────────────────
# Build a second-level model on OOF preds + key base features
print("\nFitting meta-learner (stage 2)...")
from sklearn.linear_model import Ridge

# Meta features: OOF preds + most informative base features
meta_base_cols = ['geo_time_mean', 'lag_day48', 'lag48_roll4', 'hour',
                  'geo_mean', 'time_of_day', 'd49_d48_ratio',
                  'day49_early_mean', 'd48_geo_hour_mean',
                  'lag48_ratio_adj', 'geo_hour_mean']
meta_base_idx  = [feature_cols.index(c) for c in meta_base_cols]

meta_X_train = np.hstack([oof_preds, X_train[:, meta_base_idx]])
meta_X_test  = np.hstack([test_preds, X_test[:, meta_base_idx]])

# Stage 2a: Ridge meta-learner
ridge = Ridge(alpha=1.0)
meta_oof_ridge = cross_val_predict(ridge, meta_X_train, y_train, cv=cv)
ridge.fit(meta_X_train, y_train)
meta_test_ridge = ridge.predict(meta_X_test)
print(f"Ridge meta OOF R²: {r2_score(y_train, meta_oof_ridge):.6f}")

# Stage 2b: HGB meta-learner
hgb_meta = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                          max_depth=5, min_samples_leaf=20,
                                          l2_regularization=1.0, random_state=42)
meta_oof_hgb = cross_val_predict(hgb_meta, meta_X_train, y_train, cv=cv)
hgb_meta.fit(meta_X_train, y_train)
meta_test_hgb = hgb_meta.predict(meta_X_test)
print(f"HGB meta OOF R²: {r2_score(y_train, meta_oof_hgb):.6f}")

# ET meta-learner
et_meta = ExtraTreesRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5,
                               max_features=0.7, random_state=42, n_jobs=-1)
meta_oof_et = cross_val_predict(et_meta, meta_X_train, y_train, cv=cv, n_jobs=-1)
et_meta.fit(meta_X_train, y_train)
meta_test_et = et_meta.predict(meta_X_test)
print(f"ET meta OOF R²: {r2_score(y_train, meta_oof_et):.6f}")

# Optimize blend of meta-learners
meta_oof_stack = np.stack([meta_oof_ridge, meta_oof_hgb, meta_oof_et], axis=1)
meta_test_stack = np.stack([meta_test_ridge, meta_test_hgb, meta_test_et], axis=1)

def neg_r2_meta(w):
    w = np.abs(w)/np.abs(w).sum()
    return -r2_score(y_train, meta_oof_stack @ w)

best_r2_meta = -1
best_w_meta  = None
for seed in range(10):
    rng = np.random.default_rng(seed)
    w0  = rng.dirichlet(np.ones(3))
    res = minimize(neg_r2_meta, w0, method='Nelder-Mead',
                   options={'maxiter':10000,'xatol':1e-10,'fatol':1e-10})
    w   = np.abs(res.x)/np.abs(res.x).sum()
    r2  = r2_score(y_train, meta_oof_stack @ w)
    if r2 > best_r2_meta:
        best_r2_meta = r2
        best_w_meta  = w

print(f"\nMeta blend: Ridge={best_w_meta[0]:.4f} HGB={best_w_meta[1]:.4f} ET={best_w_meta[2]:.4f}")
print(f"Final OOF R²: {best_r2_meta:.6f}")
print(f"Final estimated score: {100*best_r2_meta:.4f} / 100")

# ── Save submission ────────────────────────────────────────────────────────────
final_preds = np.clip(meta_test_stack @ best_w_meta, 0, 1)
submission  = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
submission.to_csv('submission.csv', index=False)
print(f"\nsubmission.csv  →  {submission.shape[0]} rows × {submission.shape[1]} cols")
print(submission.head())